In [34]:
# ============================================
# Standard library imports
# ============================================

import sys
import re
import ast
import logging
import warnings
from pathlib import Path

from IPython.display import display

warnings.filterwarnings("ignore")


# ============================================
# Numeric / statistics
# ============================================

import numpy as np
import pandas as pd

import scipy
from scipy import stats
from scipy.stats import (
    spearmanr,
    pearsonr,
    zscore
)


# ============================================
# Machine learning / multivariate analysis
# ============================================

from sklearn.preprocessing import StandardScaler

from sklearn.decomposition import PCA

from sklearn.cluster import (
    KMeans,
    AgglomerativeClustering
)

from sklearn.metrics import (
    silhouette_score
)

from sklearn.manifold import TSNE


# ============================================
# Distance / clustering
# ============================================

from scipy.spatial.distance import (
    pdist,
    squareform
)

from scipy.cluster.hierarchy import (
    linkage,
    dendrogram,
    fcluster
)


# ============================================
# Plotting / visualization
# ============================================

import matplotlib as mpl
import matplotlib.pyplot as plt

from matplotlib.ticker import FuncFormatter
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Ellipse

import seaborn as sns


# ============================================
# Version info
# ============================================

print(f"python  = {sys.version_info[0]}.{sys.version_info[1]}.{sys.version_info[2]}")
print(f"pandas  = {pd.__version__}")
print(f"numpy   = {np.__version__}")
print(f"scipy   = {scipy.__version__}")


# ============================================
# Logging
# ============================================

logging.root.handlers = []

stream_handler = logging.StreamHandler(sys.stderr)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)8s: %(message)s",
    handlers=[stream_handler],
)

logger = logging.getLogger(__name__)
logging.getLogger("matplotlib").setLevel(logging.WARNING)

python  = 3.11.15
pandas  = 2.3.3
numpy   = 2.4.6
scipy   = 1.17.1


In [2]:
# ============================================
# Global plotting configuration
# ============================================

_PLOT_CFG = {
    "fig_w": 6.0,
    "fig_h": 6.0,
    "dpi": 300,
}


SPECIES_INFO = {
    "AT21": {
        "label": "AT",
        "short": "AT21",
        "color": "#664D0AFF",
        "marker": "o",
    },
    "NB21": {
        "label": "NB",
        "short": "NB21",
        "color": "#7e3131",
        "marker": "^",
    },
    "OS21": {
        "label": "OS",
        "short": "OS21",
        "color": "#13563f",
        "marker": "s",
    },
}

def set_plot_style(
    *,
    base_fontsize=12,
    title_fontsize=14,
    label_fontsize=12,
    tick_fontsize=11,
    legend_fontsize=12,
    dpi=300,
    axes_linewidth=1.2,
    spines_top=True,
    spines_right=True,
    tick_size_major=6,
    tick_dir="out",
    grid=False,
    fig_w=6.0,
    fig_h=6.0,
):
    sns.set_style("ticks")

    mpl.rcParams.update({
        "font.family": "DejaVu Sans",
        "font.size": base_fontsize,

        "axes.titlesize": title_fontsize,
        "axes.labelsize": label_fontsize,

        "xtick.labelsize": tick_fontsize,
        "ytick.labelsize": tick_fontsize,

        "legend.fontsize": legend_fontsize,

        "figure.dpi": dpi,
        "savefig.dpi": dpi,

        "axes.linewidth": axes_linewidth,
        "axes.spines.top": spines_top,
        "axes.spines.right": spines_right,
        "axes.grid": grid,
        "axes.axisbelow": True,

        "xtick.major.size": tick_size_major,
        "ytick.major.size": tick_size_major,
        "xtick.direction": tick_dir,
        "ytick.direction": tick_dir,

        "legend.frameon": False,

        "savefig.bbox": "tight",
        "savefig.transparent": False,
        "figure.autolayout": False,
    })

    _PLOT_CFG.update({
        "fig_w": fig_w,
        "fig_h": fig_h,
        "dpi": dpi,
    })


def make_fig(w=None, h=None, dpi=None):
    W = float(w) if w is not None else _PLOT_CFG["fig_w"]
    H = float(h) if h is not None else _PLOT_CFG["fig_h"]
    D = dpi if dpi is not None else _PLOT_CFG["dpi"]

    fig, ax = plt.subplots(
        figsize=(W, H),
        dpi=D,
    )

    return fig, ax


def _compact_formatter():
    def _fmt(x, _pos=None):
        axx = abs(x)

        if axx >= 1e9:
            s = f"{x / 1e9:.1f}B"
        elif axx >= 1e6:
            s = f"{x / 1e6:.1f}M"
        elif axx >= 1e3:
            s = f"{x / 1e3:.1f}k"
        else:
            s = f"{x:.2g}"

        return (
            s.replace(".0B", "B")
             .replace(".0M", "M")
             .replace(".0k", "k")
        )

    return FuncFormatter(_fmt)


def format_axis(
    ax,
    *,
    xlabel=None,
    ylabel=None,
    compact_ticks=(),
):
    if xlabel is not None:
        ax.set_xlabel(xlabel)

    if ylabel is not None:
        ax.set_ylabel(ylabel)

    fmt = _compact_formatter()

    if "x" in compact_ticks:
        ax.xaxis.set_major_formatter(fmt)

    if "y" in compact_ticks:
        ax.yaxis.set_major_formatter(fmt)

    return ax


# ============================================
# Joint scatter with KDE marginals
# ============================================

def safe_pearsonr(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]

    if len(x) < 2:
        return np.nan, np.nan

    if np.std(x) == 0 or np.std(y) == 0:
        return np.nan, np.nan

    return stats.pearsonr(x, y)


def safe_spearmanr(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]

    if len(x) < 2:
        return np.nan, np.nan

    if np.std(x) == 0 or np.std(y) == 0:
        return np.nan, np.nan

    return stats.spearmanr(x, y)

def _kde_1d(values, lo, hi, num=256):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    grid = np.linspace(lo, hi, num)

    if len(values) < 2:
        return grid, np.zeros_like(grid)

    try:
        kde = stats.gaussian_kde(values)
        dens = kde(grid)
        dens /= dens.max() if dens.max() > 0 else 1
        return grid, dens

    except Exception:
        return grid, np.zeros_like(grid)


def joint_scatter(
    x,
    y,
    *,
    color=None,
    point_size=18,
    alpha=0.65,
    show_identity=True,
    show_regression=True,
    annotate=True,
    annotate_spearman=True,
    xlabel=None,
    ylabel=None,
    title=None,
    figsize=None,
    w=None,
    h=None,
    dpi=None,
    annotate_fontsize=14,
):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]

    n = len(x)

    if figsize is not None:
        FW, FH = figsize
    else:
        FW = float(w) if w is not None else _PLOT_CFG["fig_w"]
        FH = float(h) if h is not None else _PLOT_CFG["fig_h"]

    fig = plt.figure(
        figsize=(FW, FH),
        dpi=(dpi or _PLOT_CFG["dpi"]),
    )

    gs = GridSpec(
        2,
        2,
        width_ratios=(4, 1),
        height_ratios=(1, 4),
        hspace=0.05,
        wspace=0.05,
    )

    ax_top = fig.add_subplot(gs[0, 0])
    ax_joint = fig.add_subplot(gs[1, 0], sharex=ax_top)
    ax_right = fig.add_subplot(gs[1, 1], sharey=ax_joint)

    ax_joint.scatter(
        x,
        y,
        s=point_size,
        alpha=alpha,
        edgecolor="none",
        color=color,
    )

    lo = float(np.nanmin([x.min(), y.min()]))
    hi = float(np.nanmax([x.max(), y.max()]))

    pad = 0.05 * (hi - lo if hi > lo else 1.0)

    lo -= pad
    hi += pad

    ax_joint.set_xlim(lo, hi)
    ax_joint.set_ylim(lo, hi)

    if show_identity:
        ax_joint.plot(
            [lo, hi],
            [lo, hi],
            ls="--",
            lw=1.2,
            color="0.65",
            zorder=1,
        )

    if show_regression and n >= 2 and np.std(x) > 0 and np.std(y) > 0:
        slope, intercept = np.polyfit(x, y, 1)

        ax_joint.plot(
            [lo, hi],
            slope * np.array([lo, hi]) + intercept,
            color="black",
            lw=1.5,
            zorder=2,
        )

    format_axis(
        ax_joint,
        xlabel=xlabel,
        ylabel=ylabel,
        compact_ticks=(),
    )

    if title:
        ax_joint.set_title(title)

    if annotate and n >= 2 and np.std(x) > 0 and np.std(y) > 0:
        rp, _ = safe_pearsonr(x, y)
        rs, _ = safe_spearmanr(x, y)

        txt = rf"$r_p = {rp:.2f}$"

        if annotate_spearman:
            txt += "\n" + rf"$r_s = {rs:.2f}$"

        txt += f"\n$n = {n}$"

        ax_joint.text(
            0.04,
            0.96,
            txt,
            transform=ax_joint.transAxes,
            ha="left",
            va="top",
            fontsize=annotate_fontsize,
        )

    gx, dx = _kde_1d(x, lo, hi)
    gy, dy = _kde_1d(y, lo, hi)

    ax_top.plot(gx, dx, lw=2, color=color)
    ax_top.axis("off")

    ax_right.plot(dy, gy, lw=2, color=color)
    ax_right.axis("off")

    plt.tight_layout()

    return fig, (ax_joint, ax_top, ax_right)


set_plot_style()

In [3]:
# ============================================
# Load and inspect one example result file
# ============================================

RESULT_DIR = Path("../results")

example_files = {
    "lgbm_importance": RESULT_DIR / "lgbm" / "AT21.lgbm.feature_importance.tsv",
    "lgbm_shap": RESULT_DIR / "lgbm" / "AT21.lgbm.shap_summary.tsv",
    "rf_importance": RESULT_DIR / "rf" / "AT21.rf.feature_importance.tsv",
    "rf_shap": RESULT_DIR / "rf" / "AT21.rf.shap_summary.tsv",
}

for name, path in example_files.items():
    print("=" * 80)
    print(name)
    print(path)

    df = pd.read_csv(path, sep="\t")

    print("shape:", df.shape)
    print("columns:")
    print(df.columns.tolist())

    display(df.head())

lgbm_importance
../results/lgbm/AT21.lgbm.feature_importance.tsv
shape: (440, 5)
columns:
['species', 'model', 'rank', 'feature', 'importance']


,species,model,rank,feature,importance
0,AT21,LightGBM,1,5'UTR.CUU-freq,29.0
1,AT21,LightGBM,2,5'UTR.C-freq,28.0
2,AT21,LightGBM,3,5'UTR.S-freq,27.0
3,AT21,LightGBM,4,5'UTR.U-freq,26.0
4,AT21,LightGBM,5,5'UTR.MFE,25.0


lgbm_shap
../results/lgbm/AT21.lgbm.shap_summary.tsv
shape: (464, 7)
columns:
['species', 'model', 'feature', 'mean_SHAP', 'mean_abs_SHAP', 'SHAP_rank_abs', 'SHAP_rank_mean']


,species,model,feature,mean_SHAP,mean_abs_SHAP,SHAP_rank_abs,SHAP_rank_mean
0,AT21,LightGBM,CDS.G-freq,0.000714,0.040250,1.0,14.0
1,AT21,LightGBM,5'UTR.MFE,0.001383,0.033028,2.0,4.0
2,AT21,LightGBM,5'UTR.A-freq,-0.000306,0.032551,3.0,69.0
3,AT21,LightGBM,5'UTR.S-freq,0.002141,0.029270,4.0,1.0
4,AT21,LightGBM,CDS.AU-freq,-0.000246,0.027730,5.0,90.0


rf_importance
../results/rf/AT21.rf.feature_importance.tsv
shape: (450, 5)
columns:
['species', 'model', 'rank', 'feature', 'importance']


,species,model,rank,feature,importance
0,AT21,RandomForest,1,CDS.G-freq,0.042472
1,AT21,RandomForest,2,5'UTR.MFE,0.027173
2,AT21,RandomForest,3,CDS.GG-freq,0.018439
3,AT21,RandomForest,4,5'UTR.A-freq,0.016365
4,AT21,RandomForest,5,CDS.AU-freq,0.015806


rf_shap
../results/rf/AT21.rf.shap_summary.tsv
shape: (464, 7)
columns:
['species', 'model', 'feature', 'mean_SHAP', 'mean_abs_SHAP', 'SHAP_rank_abs', 'SHAP_rank_mean']


,species,model,feature,mean_SHAP,mean_abs_SHAP,SHAP_rank_abs,SHAP_rank_mean
0,AT21,RandomForest,CDS.G-freq,0.000370,0.096996,1.0,14.0
1,AT21,RandomForest,5'UTR.MFE,0.001694,0.056510,2.0,3.0
2,AT21,RandomForest,CDS.GG-freq,0.003197,0.040456,3.0,1.0
3,AT21,RandomForest,5'UTR.A-freq,0.002378,0.037820,4.0,2.0
4,AT21,RandomForest,CDS.AU-freq,0.001397,0.030959,5.0,4.0


In [35]:
# ============================================
# Load all feature importance tables
# ============================================

RESULT_DIR = Path("../results")

species_list = ["AT21", "NB21", "OS21"]

model_dirs = {
    "LGBM": "lgbm",
    "RF": "rf",
}

importance_tables = []

for model_name, model_dir in model_dirs.items():

    for sp in species_list:

        path = (
            RESULT_DIR /
            model_dir /
            f"{sp}.{model_dir}.feature_importance.tsv"
        )

        df = pd.read_csv(path, sep="\t")

        df["species"] = sp
        df["model"] = model_name

        importance_tables.append(df)

importance_all = pd.concat(
    importance_tables,
    ignore_index=True
)

print(importance_all.shape)

display(
    importance_all.head()
)

(2663, 5)


,species,model,rank,feature,importance
0,AT21,LGBM,1,5'UTR.CUU-freq,29.0
1,AT21,LGBM,2,5'UTR.C-freq,28.0
2,AT21,LGBM,3,5'UTR.S-freq,27.0
3,AT21,LGBM,4,5'UTR.U-freq,26.0
4,AT21,LGBM,5,5'UTR.MFE,25.0
